In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit


In [0]:
# Storage Account real del proyecto
storage_account_name = "stfrauddatabricks"

# Containers
container_raw = "raw"
container_bronze = "bronze"

# Carpeta base del proyecto
project_folder = "fraud_detection"

# Rutas base
raw_base_path = f"abfss://{container_raw}@{storage_account_name}.dfs.core.windows.net/{project_folder}"
bronze_base_path = f"abfss://{container_bronze}@{storage_account_name}.dfs.core.windows.net/{project_folder}"

print("RAW path    :", raw_base_path)
print("BRONZE path :", bronze_base_path)

In [0]:
transactions_raw_path = f"{raw_base_path}/transactions/PS_20174392719_1491204439457_log.csv"
transaction_type_risk_raw_path = f"{raw_base_path}/reference/transaction_type_risk.csv"
customer_profile_raw_path = f"{raw_base_path}/profiles/customer_account_profile.csv"

print("Archivo transacciones       :", transactions_raw_path)
print("Archivo riesgo por tipo     :", transaction_type_risk_raw_path)
print("Archivo perfil de cuentas   :", customer_profile_raw_path)

In [0]:
bronze_transactions_path = f"{bronze_base_path}/transactions_delta"
bronze_transaction_type_risk_path = f"{bronze_base_path}/transaction_type_risk_delta"
bronze_customer_profile_path = f"{bronze_base_path}/customer_account_profile_delta"

print("Salida Bronze transacciones      :", bronze_transactions_path)
print("Salida Bronze riesgo por tipo    :", bronze_transaction_type_risk_path)
print("Salida Bronze perfil de cuentas  :", bronze_customer_profile_path)

In [0]:
df_transactions_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(transactions_raw_path)
)

df_transactions_bronze = (
    df_transactions_raw
    .withColumn("source_file", input_file_name())
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("bronze_layer", lit("bronze_transactions"))
)

display(df_transactions_bronze.limit(10))

In [0]:
df_transaction_type_risk_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(transaction_type_risk_raw_path)
)

df_transaction_type_risk_bronze = (
    df_transaction_type_risk_raw
    .withColumn("source_file", input_file_name())
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("bronze_layer", lit("bronze_transaction_type_risk"))
)

display(df_transaction_type_risk_bronze)

In [0]:


df_customer_profile_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(customer_profile_raw_path)
)

df_customer_profile_bronze = (
    df_customer_profile_raw
    .withColumn("source_file", input_file_name())
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("bronze_layer", lit("bronze_customer_account_profile"))
)

display(df_customer_profile_bronze.limit(10))

In [0]:

print("Cantidad de registros cargados en Bronze:")

print("Transacciones       :", df_transactions_bronze.count())
print("Riesgo por tipo     :", df_transaction_type_risk_bronze.count())
print("Perfil de cuentas   :", df_customer_profile_bronze.count())

In [0]:

(
    df_transactions_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(bronze_transactions_path)
)

(
    df_transaction_type_risk_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(bronze_transaction_type_risk_path)
)

(
    df_customer_profile_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(bronze_customer_profile_path)
)

print("Datasets guardados correctamente en formato Delta en la capa Bronze.")

In [0]:
df_transactions_bronze_check = spark.read.format("delta").load(bronze_transactions_path)
df_transaction_type_risk_bronze_check = spark.read.format("delta").load(bronze_transaction_type_risk_path)
df_customer_profile_bronze_check = spark.read.format("delta").load(bronze_customer_profile_path)

print("Validación de tablas Delta Bronze:")
print("Transacciones Delta       :", df_transactions_bronze_check.count())
print("Riesgo por tipo Delta     :", df_transaction_type_risk_bronze_check.count())
print("Perfil de cuentas Delta   :", df_customer_profile_bronze_check.count())

In [0]:
print("Esquema Bronze - Transacciones")
df_transactions_bronze_check.printSchema()

print("Esquema Bronze - Riesgo por tipo")
df_transaction_type_risk_bronze_check.printSchema()

print("Esquema Bronze - Perfil de cuentas")
df_customer_profile_bronze_check.printSchema()